In [2]:
from langchain_google_genai import GoogleGenerativeAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

In [3]:
class BatsManState(TypedDict):
    runs: int
    balls: int
    fours: int
    sixes: int
    
    strike_rate: float
    boundaryperball: float
    boundary_percent: float
    summary: str

In [17]:
def strike_rate(state: BatsManState):
    return {
        "strike_rate":
            (state['runs'] / state['balls']) * 100
    }

def boundaryperball(state: BatsManState):
    return {
        "boundaryperball":
            (state["fours"] + state["sixes"]) / state["balls"]
    }

def boundary_percent(state: BatsManState):
    return {
        "boundary_percent":
            ((state['fours'] + state['sixes']) / state['runs']) * 100
    }

def summary(state: BatsManState):
    return {
        "summary": f"""
        Runs: {state['runs']}
        Balls: {state['balls']}
        Fours: {state['fours']}
        Sixes: {state['sixes']}
        Strike Rate: {state['strike_rate']}
        Boundary per Ball: {state['boundaryperball']}
        Boundary Percent: {state['boundary_percent']}
        """
    }

In [18]:
graph = StateGraph(BatsManState)

graph.add_node('calclate_strike_rate', strike_rate)
graph.add_node('calculate_boundaryperball', boundaryperball)
graph.add_node('calculate_boundary_percent', boundary_percent)
graph.add_node('summary', summary)

# Add edges to the graph to define the workflow

graph.add_edge(START, 'calclate_strike_rate')
graph.add_edge(START, 'calculate_boundaryperball')
graph.add_edge(START, 'calculate_boundary_percent')

graph.add_edge('calclate_strike_rate', 'summary')
graph.add_edge('calculate_boundaryperball', 'summary')
graph.add_edge('calculate_boundary_percent', 'summary')

graph.add_edge('summary', END)


workflow = graph.compile()

In [19]:
initial_state = BatsManState(
    runs=100,
    balls=20,
    fours=6,
    sixes=4,
    strike_rate=0.0,
    boundaryperball=0.0,
    boundary_percent=0.0,
    summary=""
)

final_state = workflow.invoke(initial_state)
print(final_state['summary'])


        Runs: 100
        Balls: 20
        Fours: 6
        Sixes: 4
        Strike Rate: 500.0
        Boundary per Ball: 0.5
        Boundary Percent: 10.0
        
